In [ ]:
import re
from html.parser import HTMLParser

html_path = r'C:\Projects\aerojet-academy\docs\html\Internal Exam System — Full UI & Architecture Plan · Aerojet Academy.html'
md_path = r'C:\Projects\aerojet-academy\docs\plans\internal-exam-system-ui-plan.md'

with open(html_path, 'r', encoding='utf-8') as f:
    html = f.read()

# Extract main content
main_match = re.search(r'<main class="he-main">([\s\S]*)</main>', html)
if not main_match:
    raise SystemExit('Could not find <main> content')

content = main_match.group(1)

# Remove nav, aside.toc, footer
content = re.sub(r'<nav[\s\S]*?</nav>', '', content)
content = re.sub(r'<aside class="he-toc"[\s\S]*?</aside>', '', content)
content = re.sub(r'<footer[\s\S]*?</footer>', '', content)

def decode_html(s):
    return s.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"').replace('&#39;', "'").replace('&nbsp;', ' ')

# Convert header
header_match = re.search(r'<header>([\s\S]*?)</header>', content)
header_md = ''
if header_match:
    header_content = header_match.group(1)
    title_match = re.search(r'<h1[^>]*>(.*?)</h1>', header_content)
    title = decode_html(title_match.group(1)) if title_match else 'Internal Exam System — Full UI & Architecture Plan'
    header_md = f'# {title}\n\n'
    
    for p in re.findall(r'<p[^>]*>([\s\S]*?)</p>', header_content):
        text = decode_html(p)
        if text.strip():
            header_md += f'{text}\n\n'
    content = content.replace(header_match.group(0), '')

content = content.strip()

# Extract code blocks
code_blocks = []
def extract_code_blocks(m):
    lang = m.group(1) or ''
    code = decode_html(m.group(2).strip())
    idx = len(code_blocks)
    code_blocks.append((lang, code))
    return f'%%CODEBLOCK_{idx}%%'

content = re.sub(r'<pre><code(?: class="language-(\w+)")?>([\s\S]*?)</code></pre>', extract_code_blocks, content)

# Inline elements
content = re.sub(r'<code>(.*?)</code>', lambda m: f'`{decode_html(m.group(1))}`', content)
content = re.sub(r'<strong>(.*?)</strong>', lambda m: f'**{m.group(1)}**', content)
content = re.sub(r'<em>(.*?)</em>', lambda m: f'*{m.group(1)}*', content)
content = re.sub(r'<del>(.*?)</del>', lambda m: f'~~{m.group(1)}~~', content)
content = re.sub(r'<a href="([^"]*)">(.*?)</a>', lambda m: f'[{decode_html(m.group(2))}]({m.group(1)})', content)

# HR
content = re.sub(r'<hr\s*/>', '\n---\n', content)

# Blockquotes
content = re.sub(r'<blockquote class="he-pull">([\s\S]*?)</blockquote>', lambda m: '\n' + '\n'.join([f'> {line}' for line in decode_html(m.group(1).replace('<p>', '').replace('</p>', '')).strip().split('\n')]) + '\n', content)
content = re.sub(r'<blockquote>([\s\S]*?)</blockquote>', lambda m: '\n' + '\n'.join([f'> {line}' for line in decode_html(m.group(1).replace('<p>', '').replace('</p>', '')).strip().split('\n')]) + '\n', content)

# Tables
def convert_table(m):
    table_content = m.group(1)
    rows = []
    thead_match = re.search(r'<thead>([\s\S]*?)</thead>', table_content)
    tbody_match = re.search(r'<tbody>([\s\S]*?)</tbody>', table_content)
    
    header_cells = []
    if thead_match:
        header_row = re.search(r'<tr>([\s\S]*?)</tr>', thead_match.group(1))
        if header_row:
            header_cells = re.findall(r'<(th|td)(?: align="[^"]*")?>([\s\S]*?)</\1>', header_row.group(1))
            header_cells = [decode_html(c[1].strip()) for c in header_cells]
    
    if not header_cells and tbody_match:
        first_row = re.search(r'<tr>([\s\S]*?)</tr>', tbody_match.group(1))
        if first_row:
            header_cells = re.findall(r'<(th|td)(?: align="[^"]*")?>([\s\S]*?)</\1>', first_row.group(1))
            header_cells = [decode_html(c[1].strip()) for c in header_cells]
    
    if header_cells:
        rows.append('| ' + ' | '.join(header_cells) + ' |')
        rows.append('| ' + ' | '.join(['---'] * len(header_cells)) + ' |')
    
    if tbody_match:
        data_rows = re.findall(r'<tr>([\s\S]*?)</tr>', tbody_match.group(1))
        if not thead_match and header_cells:
            data_rows = data_rows[1:]  # Skip first row if used as header
        
        for row in data_rows:
            cells = re.findall(r'<(th|td)(?: align="[^"]*")?>([\s\S]*?)</\1>', row)
            cells = [decode_html(c[1].strip()) for c in cells]
            rows.append('| ' + ' | '.join(cells) + ' |')
    
    return '\n' + '\n'.join(rows) + '\n'

content = re.sub(r'<table>([\s\S]*?)</table>', convert_table, content)

# Sections
content = re.sub(r'<section class="he-section" id="([^"]*)">([\s\S]*?)</section>', lambda m: convert_section(m.group(2)), content)

def convert_section(inner):
    num_match = re.search(r'<div class="he-section__num">(.*?)</div>', inner)
    title_match = re.search(r'<h2[^>]*>(.*?)</h2>', inner)
    
    heading = ''
    if num_match:
        num_text = decode_html(num_match.group(1))
        clean_num = re.sub(r'^\d+\s*·\s*', '', num_text)
        heading = f'## {clean_num}'
    elif title_match:
        heading = f'## {decode_html(title_match.group(1))}'
    
    rest = re.sub(r'<div class="he-section__num">[\s\S]*?</div>', '', inner)
    rest = re.sub(r'<h2[^>]*>[\s\S]*?</h2>', '', rest)
    rest = rest.strip()
    
    if rest:
        return heading + '\n\n' + rest
    return heading

# h3, h4
content = re.sub(r'<h3[^>]*>(.*?)</h3>', lambda m: f'\n### {decode_html(m.group(1))}\n', content)
content = re.sub(r'<h4[^>]*>(.*?)</h4>', lambda m: f'\n#### {decode_html(m.group(1))}\n', content)

# Task list items
content = re.sub(r'<li class="task-list-item"><input type="checkbox"[^>]*?(checked)?[^>]*>([\s\S]*?)</li>', 
                 lambda m: f'- [{"x" if m.group(1) else " "}] {decode_html(m.group(2).replace("<li class=\"task-list-item\">", "").replace("</li>", "").strip())}\n', content)

# UL lists
prev_content = ''
while prev_content != content:
    prev_content = content
    content = re.sub(r'<ul>([\s\S]*?)</ul>', lambda m: '\n' + '\n'.join([
        f'- {decode_html(item.replace("<li>", "").replace("</li>", "").strip())}'
        for item in re.findall(r'<li>([\s\S]*?)</li>', m.group(1))
    ]) + '\n', content)

# OL lists
prev_content = ''
while prev_content != content:
    prev_content = content
    content = re.sub(r'<ol>([\s\S]*?)</ol>', lambda m: '\n' + '\n'.join([
        f'{i+1}. {decode_html(item.replace("<li>", "").replace("</li>", "").strip())}'
        for i, item in enumerate(re.findall(r'<li>([\s\S]*?)</li>', m.group(1)))
    ]) + '\n', content)

# p tags
content = re.sub(r'<p[^>]*>([\s\S]*?)</p>', lambda m: f'{decode_html(m.group(1))}\n\n', content)

# Remaining divs
content = re.sub(r'<div[^>]*>([\s\S]*?)</div>', lambda m: m.group(1), content)

# Restore code blocks
content = re.sub(r'%%CODEBLOCK_(\d+)%%', lambda m: f'```{code_blocks[int(m.group(1))][0]}\n{code_blocks[int(m.group(1))][1]}\n```' if code_blocks[int(m.group(1))][0] else f'```\n{code_blocks[int(m.group(1))][1]}\n```', content)

# Clean excessive whitespace
content = re.sub(r'\n{3,}', '\n\n', content)
content = content.strip()

markdown = header_md + '\n' + content
markdown = re.sub(r'^\s+', '', markdown)
markdown = re.sub(r'\s+$', '', markdown)

with open(md_path, 'w', encoding='utf-8') as f:
    f.write(markdown)

print(f'Written {len(markdown.splitlines())} lines to {md_path}')
